# <center>My First ML Project: Predicting Accident History of Used Cars</center>

## Project Overview

This is my first end-to-end classification project. I'm reusing the cleaned/engineered
`used_cars.csv` dataset from my earlier price-prediction notebook, but this time the goal
is different: instead of predicting the **price** of a car (a regression problem), I want
to predict whether a car has an **accident history** (a binary classification problem) —
`1` = at least one accident or damage reported, `0` = none reported.

I'm also trying to follow good software practices this time, based on a training script
(`training.py`) a more experienced teammate shared with me. So this notebook is structured
around the same ideas:
- **Config via environment variables** (`.env` file) instead of hard-coded values
- **Logging** instead of scattered `print()` statements for status messages
- A reusable **metrics helper function**
- A single **Pipeline** (preprocessing + model) tuned with **GridSearchCV**
- Clear **train vs. test** evaluation to check for overfitting
- Saving the final tuned model with **joblib**

## Project Objective
- **Data Cleaning**: Reuse the cleaning logic from my first notebook (fix data types, fill missing values, standardize categories, remove outliers).
- **Feature Engineering**: Extract `horse_power`, `engine_capacity`, and `cylinder_number` from the `engine` text field, and convert `model_year` into `age`.
- **Target Definition**: Turn the `accident` column into a binary target (`had_accident`).
- **Preprocessing + Pipeline**: Scale numeric features, one-hot encode categorical features.
- **Model Training**: Logistic Regression, tuned via `GridSearchCV`.
- **Evaluation**: Accuracy, precision, recall, F1-score, confusion matrix, ROC curve.
- **Final Product**: A saved, reusable classification pipeline (`.pkl` file).

## 1. Importing Libraries

In [9]:
#=============================================================================
# All the libraries used in this file
#=============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from joblib import dump
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_curve,
    roc_auc_score
)
from dotenv import load_dotenv
load_dotenv()
import logging

## 2. Logging Configuration

Instead of relying only on `print()`, I'm setting up basic logging like in `training.py`.
This makes it easier to see timestamps and severity levels for each step, which is helpful
once the notebook grows longer.

In [10]:
# =============================================================================
# logging configuration
# =============================================================================

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

## 3. Configuration

`training.py` pulls all its settings (dataset path, model output path, target column,
test size, random state) from a `.env` file via `os.getenv`. I'm doing the same thing here
for consistency — but since this is a notebook meant to run standalone (without a real
`.env` file present), I fall back to sensible defaults whenever an environment variable
isn't set.

In [11]:
DATASET = os.getenv("DATASET", "/kaggle/input/used-car-price-prediction-dataset/used_cars.csv")
MODEL_PATH = os.getenv("MODEL_PATH", "Used_Car_Accident_Pipeline.pkl")
TARGET_COLUMN = os.getenv("TARGET_COLUMN", "had_accident")
TEST_SIZE = float(os.getenv("TEST_SIZE", "0.2"))
RANDOM_STATE = int(os.getenv("RANDOM_STATE", "42"))

logger.info(f"Dataset path: {DATASET}")
logger.info(f"Model will be saved to: {MODEL_PATH}")
logger.info(f"Target column: {TARGET_COLUMN}")
logger.info(f"Test size: {TEST_SIZE}, Random state: {RANDOM_STATE}")

2026-08-16 13:31:30,891 - INFO - Dataset path: /kaggle/input/used-car-price-prediction-dataset/used_cars.csv
2026-08-16 13:31:30,894 - INFO - Model will be saved to: Used_Car_Accident_Pipeline.pkl
2026-08-16 13:31:30,897 - INFO - Target column: had_accident
2026-08-16 13:31:30,899 - INFO - Test size: 0.2, Random state: 42


## 4. Loading the Dataset

In [12]:
# =============================================================================
# Dataset
# =============================================================================

car_data = pd.read_csv(r"\Used_Car_Price_Prerdiction\used_cars.csv")
print(car_data.head(5))
logger.info(f"Dataset Shape: {car_data.shape}")

2026-08-16 13:31:30,958 - INFO - Dataset Shape: (4009, 12)


      brand                            model  model_year      milage  \
0      Ford  Utility Police Interceptor Base        2013  51,000 mi.   
1   Hyundai                     Palisade SEL        2021  34,742 mi.   
2     Lexus                    RX 350 RX 350        2022  22,372 mi.   
3  INFINITI                 Q50 Hybrid Sport        2015  88,900 mi.   
4      Audi        Q3 45 S line Premium Plus        2021   9,835 mi.   

       fuel_type                                             engine  \
0  E85 Flex Fuel  300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...   
1       Gasoline                               3.8L V6 24V GDI DOHC   
2       Gasoline                                     3.5 Liter DOHC   
3         Hybrid  354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...   
4       Gasoline                         2.0L I4 16V GDI DOHC Turbo   

        transmission                 ext_col int_col  \
0        6-Speed A/T                   Black   Black   
1  8-Speed Automatic        

## 5. Data Cleaning

I'm reusing the same cleaning steps I worked out in my first notebook, since this is the
same raw dataset. Quick recap of what needs fixing:
- `milage` and `price` are stored as text (`"51,000 mi."`, `"$10,300"`) instead of numbers.
- `clean_title` has missing values that should be treated as `'No'` before converting to boolean.
- `fuel_type` and `accident` have a few missing values, filled with the most common category (mode).
- `transmission` has many inconsistent labels for the same category (e.g. `"8-SPEED A/T"` vs `"Automatic, 8-Spd"`).
- `engine` uses `'–'` for electric cars instead of a real engine description.
- `milage` and `price` both contain extreme outliers that should be removed with the IQR method.

In [13]:
# Fixing inconsistent data types
car_data['milage'] = car_data['milage'].replace([',', 'mi.'], ['', ''], regex=True).astype(int)
car_data['price'] = car_data['price'].str.replace(r'[$,]', '', regex=True).astype(int)

# Filling out missing values
car_data['fuel_type'] = car_data['fuel_type'].fillna(value=car_data['fuel_type'].mode()[0])
car_data['accident'] = car_data['accident'].fillna(value=car_data['accident'].mode()[0])
car_data['clean_title'] = car_data['clean_title'].fillna('No')
car_data['clean_title'] = car_data['clean_title'].replace(['Yes', 'No'], [True, False])

logger.info("Missing values handled and data types fixed.")

C:\Users\ASUS\AppData\Local\Temp\ipykernel_21804\3953420660.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  car_data['clean_title'] = car_data['clean_title'].replace(['Yes', 'No'], [True, False])
2026-08-16 13:31:31,030 - INFO - Missing values handled and data types fixed.


In [14]:
# Fixing inconsistent categories

# fuel_type: '–' means fuel is not supported
car_data['fuel_type'] = car_data['fuel_type'].replace('–', 'not supported', regex=True)

# transmission: drop rows with unknown transmission, then standardize the rest
car_data = car_data[car_data['transmission'] != '–']
car_data['transmission'] = (car_data['transmission']
                             .str.lower().str.replace(r'[^a-z0-9\s]', '', regex=True)
                             .str.replace(r'\s+', ' ', regex=True)
                             .str.strip())

def format_transmission(raw_data):
    if 'automatic' in raw_data or 'at' in raw_data:
        return 'Automatic'
    elif 'manual' in raw_data or 'mt' in raw_data:
        return 'Manual'
    elif 'cvtf' in raw_data or 'cvt' in raw_data:
        return 'CVT'
    else:
        return 'Other'

car_data['transmission'] = car_data['transmission'].apply(format_transmission)

# engine: '–' means electric (no combustion engine)
car_data['engine'] = car_data['engine'].str.replace('–', 'Electric')

logger.info(f"Transmission categories after cleaning: {car_data['transmission'].unique()}")

2026-08-16 13:31:31,099 - INFO - Transmission categories after cleaning: ['Automatic' 'Other' 'Manual' 'CVT']


In [15]:
# Removing outliers (IQR method) from milage and price
numeric_columns = ['milage', 'price']
for column in numeric_columns:
    Q1 = car_data[column].quantile(0.25)
    Q3 = car_data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_whisker = Q1 - 2 * IQR
    upper_whisker = Q3 + 2 * IQR
    car_data = car_data[(car_data[column] >= lower_whisker) & (car_data[column] <= upper_whisker)]

car_data.reset_index(inplace=True, drop=True)
logger.info(f"Shape after outlier removal: {car_data.shape}")

2026-08-16 13:31:31,134 - INFO - Shape after outlier removal: (3788, 12)


## 6. Feature Engineering

Same as before: pull `horse_power`, `engine_capacity`, and `cylinder_number` out of the
free-text `engine` column, fix an engine-capacity data-entry issue (values > 10 are almost
certainly a misplaced decimal point), and convert `model_year` into `age`.

In [16]:
# Extracting numeric features from the engine text
car_data['horse_power'] = car_data['engine'].str.extract(r'(\d+(?:\.\d+)?)\s*HP').astype(float)
car_data['engine_capacity'] = car_data['engine'].str.extract(r'(\d+(?:\.\d)?)\s*L').astype(float)
car_data['cylinder_number'] = car_data['engine'].str.extract(r'(\d+(?:\.\d)?)\s*Cylinder')
car_data['cylinder_number'] = car_data['cylinder_number'].replace(np.nan, 0).astype(int)

# Fixing an obvious data-entry error (engine capacities > 10L are almost certainly a decimal slip)
car_data.loc[car_data['engine_capacity'] > 10, 'engine_capacity'] = (
    car_data.loc[car_data['engine_capacity'] > 10, 'engine_capacity'] / 10
)

# Converting model year into age
car_data['age'] = 2025 - car_data['model_year']

logger.info("Feature engineering complete: horse_power, engine_capacity, cylinder_number, age.")

2026-08-16 13:31:31,200 - INFO - Feature engineering complete: horse_power, engine_capacity, cylinder_number, age.


In [17]:
# Dropping columns that won't help the model (high-cardinality identifiers, redundant/raw text fields)
car_data = car_data.drop(columns=['model', 'ext_col', 'int_col', 'brand', 'engine', 'model_year'])
car_data.head()

,milage,fuel_type,transmission,accident,clean_title,price,horse_power,engine_capacity,cylinder_number,age
0,51000,E85 Flex Fuel,Automatic,At least 1 accident or damage reported,True,10300,300.0,3.7,6,12
1,34742,Gasoline,Automatic,At least 1 accident or damage reported,True,38005,NaN,3.8,0,4
2,22372,Gasoline,Automatic,None reported,False,54598,NaN,3.5,0,3
3,88900,Hybrid,Automatic,None reported,True,15500,354.0,3.5,6,10
4,9835,Gasoline,Automatic,None reported,False,34999,NaN,2.0,0,4


## 7. Defining the Target Variable

`training.py` expects a ready-made binary target column. Mine doesn't exist yet — I need to
derive it from `accident`, which currently holds text values like `"None reported"` and
`"At least 1 accident or damage reported"`. I'll convert this into a `0`/`1` column called
`had_accident` and drop the original text column so it isn't accidentally used as an input
feature (that would be data leakage, since it's just a reworded version of the target).

In [18]:
car_data[TARGET_COLUMN] = (car_data['accident'] != 'None reported').astype(int)
car_data = car_data.drop(columns=['accident'])

print(car_data[TARGET_COLUMN].value_counts(normalize=True))

had_accident
0    0.744456
1    0.255544
Name: proportion, dtype: float64


## 8. Train / Test / Splitting

In [19]:
# =============================================================================
# Training / Testing / Splitting
# =============================================================================

X = car_data.drop(columns=[TARGET_COLUMN])
y = car_data[TARGET_COLUMN]
logger.info(f"Feature Matrix Shape: {X.shape}")
logger.info(f"Target Vector Shape: {y.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

def print_metrics(title, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(title)
    print(f"  Accuracy: {acc:.2%}")
    print(f"  Precision: {prec:.2%}")
    print(f"  Recall: {rec:.2%}")
    print(f"  F1-score: {f1:.2%}")

2026-08-16 13:31:31,295 - INFO - Feature Matrix Shape: (3788, 9)
2026-08-16 13:31:31,298 - INFO - Target Vector Shape: (3788,)


## 9. Pipeline

One difference from `training.py`: my feature matrix `X` has both numeric columns
(`milage`, `price`, `horse_power`, `engine_capacity`, `cylinder_number`, `age`) and
categorical columns (`fuel_type`, `transmission`, `clean_title`). `training.py`'s pipeline
only had a `StandardScaler` because its input was already fully numeric. I need a
`ColumnTransformer` first, to scale the numeric columns and one-hot encode the categorical
ones, before they reach the classifier.

In [20]:
#=============================================================================
# Preprocessing: numeric vs categorical columns
#=============================================================================

numeric_features = X.select_dtypes(include=[int, float]).columns.to_list()
categorical_features = X.select_dtypes(include=['object', 'bool']).columns.to_list()

logger.info(f"Numeric features: {numeric_features}")
logger.info(f"Categorical features: {categorical_features}")

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=RANDOM_STATE, max_iter=1000))
])

2026-08-16 13:31:31,329 - INFO - Numeric features: ['milage', 'price', 'horse_power', 'engine_capacity', 'cylinder_number', 'age']
2026-08-16 13:31:31,332 - INFO - Categorical features: ['fuel_type', 'transmission', 'clean_title']


## 10. Hyperparameter Tuning with GridSearchCV

Same idea as `training.py`: search over a small grid of `C` (regularization strength) and
`solver` values, using 5-fold cross-validation to pick the best combination.

In [21]:
#=============================================================================
# Define the hyperparameter grid to search
# The keys are prefixed with the name of the pipeline step ('classifier')
#=============================================================================

param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10, 100],
    'classifier__solver': ['liblinear', 'lbfgs']
}

#=============================================================================
# Set up GridSearchCV
# cv=5 means 5-fold cross-validation
#=============================================================================

grid_search = GridSearchCV(pipeline, param_grid, cv=5, verbose=1, n_jobs=-1)

## 11. Training the Model

In [22]:
#=============================================================================
# training the model
#=============================================================================

logger.info("Starting hyperparameter tuning with GridSearchCV...")
grid_search.fit(X_train, y_train)

logger.info("Best hyperparameters found:")
print(grid_search.best_params_)

2026-08-16 13:31:31,374 - INFO - Starting hyperparameter tuning with GridSearchCV...


Fitting 5 folds for each of 10 candidates, totalling 50 fits


ValueError: 
All the 50 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
50 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\ASUS\AppData\Roaming\Python\Python314\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ASUS\AppData\Roaming\Python\Python314\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "C:\Users\ASUS\AppData\Roaming\Python\Python314\site-packages\sklearn\pipeline.py", line 663, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Use

## 12. Evaluating the Model

I'm checking both train and test metrics — like in my first notebook — to make sure the
model isn't overfitting. If train performance is much higher than test performance, that's
a sign the model has memorized the training data instead of learning general patterns.

In [ ]:
y_train_pred = grid_search.predict(X_train)
y_test_pred = grid_search.predict(X_test)

y_test_prob = grid_search.predict_proba(X_test)[:, 1]

print_metrics("TRAIN METRICS", y_train, y_train_pred)
print()
print_metrics("TEST METRICS", y_test, y_test_pred)

cm = confusion_matrix(y_test, y_test_pred)
print("\nConfusion Matrix")
print(cm)

## 13. Visualization

### Confusion Matrix

Shows how many cars were correctly/incorrectly classified as having an accident history.

In [ ]:
#=============================================================================
# Visualisation
#=============================================================================

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
            xticklabels=["Predicted 0", "Predicted 1"],
            yticklabels=["Actual 0", "Actual 1"])
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Confusion Matrix (Test Data)")
plt.show()

### ROC Curve

`training.py` imports `roc_curve` and `roc_auc_score` but doesn't plot them — since this is
a learning project, I wanted to actually use them and see how well the model separates the
two classes across different probability thresholds. A curve that hugs the top-left corner
(AUC close to 1.0) means a stronger classifier; a diagonal line (AUC = 0.5) means the model
is no better than random guessing.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_test_prob)
auc_score = roc_auc_score(y_test, y_test_prob)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, color='blue', label=f'ROC curve (AUC = {auc_score:.3f})')
plt.plot([0, 1], [0, 1], color='black', linestyle='--', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve (Test Data)')
plt.legend()
plt.show()

## 14. Saving the Model

In [ ]:
#=============================================================================
# Dumping
#=============================================================================

dump(grid_search, MODEL_PATH)
logger.info(f"Model saved to {MODEL_PATH}")

## Findings and Results

Reusing the cleaned and engineered used-car dataset, I reframed the problem from price
prediction (regression) to accident-history prediction (binary classification), and trained
a Logistic Regression model tuned with `GridSearchCV` over regularization strength (`C`)
and solver type.

The confusion matrix and precision/recall/F1 scores show how well the model separates cars
with accident history from those without, and the ROC/AUC curve shows how that trade-off
changes across different decision thresholds. Comparing train vs. test metrics also gives a
quick sanity check for overfitting.

## Things Learned

This project taught me how to restructure a notebook the way a real training script is
often organized: pulling configuration out into environment variables instead of
hard-coding values, using `logging` instead of scattered print statements, wrapping
preprocessing and the model together in a single `Pipeline`, and tuning it properly with
`GridSearchCV` instead of guessing hyperparameters by hand. I also got to work with
classification metrics (precision, recall, F1, ROC/AUC) for the first time, instead of the
regression metrics (RMSE, R²) I used in my price-prediction notebook.